# QuoridorBot: train an ML agent to beat the Depth-4 alpha-beta bot

This notebook trains a **policy-value network + MCTS** agent (AlphaZero-style) against the
existing Depth-4 alpha-beta bot and **stops automatically once it wins ≥ 95%** of evaluation
games. That result is then confirmed on a second set of fresh openings.

**How to use**
1. *Runtime → Change runtime type*: a **GPU** (T4 is fine) speeds up the network training step.
   Games are played on the **CPU**, so more CPU cores = faster training. Colab Pro / High-RAM
   runtimes have more cores.
2. Run the cells from top to bottom. Checkpoints go to **Google Drive**, so a disconnect loses at
   most one iteration.
3. If the session ends (Colab free sessions last up to ~12 h), reconnect and run all cells again:
   training **resumes** from `latest.pt` in the same run folder.

Files written to the run folder: `latest.pt`, `best.pt`, `milestone_95.pt` (when the target is
reached), `metrics.jsonl`, `buffer.npz`, `tb/` (TensorBoard).

## 1. Settings

In [ ]:
# --- Edit these if you like ------------------------------------------------
REPO_URL   = "https://github.com/eifel08/QuoridorBot.git"
BRANCH     = "feature/ml-training-pipeline"   # branch with the training code
RUN_NAME   = "vs-depth4"                      # a new name starts a new run; same name resumes
USE_DRIVE  = True                             # keep checkpoints on Google Drive (recommended)
PRESET     = "full"                           # "full", "colab-free" (lighter) or "smoke" (2-min test)
MAX_HOURS  = 11.5                             # stop cleanly before the Colab session limit
TARGET     = 0.95                             # stop when the agent wins this share vs Depth-4
EXTRA_ARGS = ""                               # e.g. "--sims 96 --games-per-iter 64"

## 2. Hardware check

In [ ]:
import os, subprocess
print("CPU cores:", os.cpu_count())
try:
    print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True).stdout or "No GPU (CPU-only runtime)")
except FileNotFoundError:
    print("No GPU (CPU-only runtime)")

## 3. Google Drive (checkpoints survive disconnects)

In [ ]:
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    RUN_DIR = f"/content/drive/MyDrive/QuoridorBot/runs/{RUN_NAME}"
else:
    RUN_DIR = f"/content/runs/{RUN_NAME}"
os.makedirs(RUN_DIR, exist_ok=True)
print("Run folder:", RUN_DIR)

## 4. Get the code and install dependencies

In [ ]:
%cd /content
if not os.path.exists("QuoridorBot"):
    !git clone --branch {BRANCH} {REPO_URL} QuoridorBot
else:
    !git -C QuoridorBot fetch origin {BRANCH} && git -C QuoridorBot checkout {BRANCH} && git -C QuoridorBot pull --ff-only origin {BRANCH}
%cd /content/QuoridorBot
!git log --oneline -1
# PyTorch and numpy are preinstalled on Colab; this only adds what is missing.
!pip install -q -r requirements-ml.txt

## 5. Quick self-test (≈1 minute)
Runs the unit tests for the training code and a tiny end-to-end training run in a scratch folder.
If this fails, nothing below will work.

In [ ]:
!python -m unittest tests.test_ml tests.test_bot 2>&1 | tail -3
!python -m quoridor.ml.train --out /content/smoke_test --preset smoke --fresh 2>&1 | tail -4

## 6. Live monitoring (TensorBoard)
Start this **before** the training cell; it refreshes while training runs. Useful charts:
- `eval/win_rate`: win rate against the Depth-4 bot, the number that has to reach 0.95;
- `train/train_win_rate`: wins in the exploratory training games;
- `train/policy_loss`, `train/value_loss`.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir "{RUN_DIR}/tb"

## 7. Train
Keep this cell running. It prints one line per iteration and an `EVAL` line every few iterations.
It stops by itself when the target is confirmed (`*** MILESTONE ... ***`), when `MAX_HOURS` is
reached, or when you press *Stop*. Re-running it continues the same run.

In [ ]:
workers = os.cpu_count()
!python -m quoridor.ml.train --out "{RUN_DIR}" --preset {PRESET} --workers {workers} --max-hours {MAX_HOURS} --target-win-rate {TARGET} {EXTRA_ARGS}

## 8. Progress plot (works any time, also after a disconnect)

In [ ]:
import json
import matplotlib.pyplot as plt

rows = [json.loads(l) for l in open(f"{RUN_DIR}/metrics.jsonl")]
ev = [r for r in rows if r["kind"] == "eval"]
tr = [r for r in rows if r["kind"] == "train"]
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot([r["iteration"] for r in ev], [r["win_rate"] for r in ev], "o-", label="eval win rate")
ax[0].plot([r["iteration"] for r in ev], [r["wilson_low"] for r in ev], ":", label="95% CI lower bound")
ax[0].axhline(TARGET, color="red", ls="--", label=f"target {TARGET:.0%}")
ax[0].set_ylim(0, 1); ax[0].set_xlabel("iteration"); ax[0].set_title("vs Depth-4 alpha-beta"); ax[0].legend()
ax[1].plot([r["iteration"] for r in tr], [r["policy_loss"] for r in tr], label="policy loss")
ax[1].plot([r["iteration"] for r in tr], [r["value_loss"] for r in tr], label="value loss")
ax[1].set_xlabel("iteration"); ax[1].legend(); ax[1].set_title("training loss")
plt.show()
if ev:
    best = max(ev, key=lambda r: r["win_rate"])
    print(f"best eval so far: {best['win_rate']:.1%} at iteration {best['iteration']} "
          f"({best['games_played']} training games)")
print("milestone reached:", any(r["kind"] == "milestone" for r in rows))

## 9. Independent check of the final model
Plays fresh openings (a different seed from training's evaluation) against the Depth-4 bot.

In [ ]:
import glob
milestones = sorted(glob.glob(f"{RUN_DIR}/milestone_*.pt"))
model = milestones[-1] if milestones else f"{RUN_DIR}/best.pt"
print("Evaluating", model)
!python -m quoridor.ml.evaluate --model "{model}" --games 200 --seed 99 --workers {os.cpu_count()}

## 10. Download the model
Then play against it locally:
`python main.py --ml-model milestone_95.pt` (desktop window) or add `--cli` for the terminal.

In [ ]:
from google.colab import files
files.download(model)